# System-Level Energy Optimization

A simplified combined thermal system: a power cycle coupled with a waste heat recovery (WHR) unit. We use `scipy.optimize.minimize` to find optimal operating points that minimize fuel consumption while meeting thermal constraints.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.optimize import minimize, NonlinearConstraint

In [ ]:
def system_model(T_turbine_in, f_recovery, T_ambient=300.0):
    eta_turbine = 0.45 * (1 - T_ambient / T_turbine_in)
    T_preheat = T_ambient + f_recovery * (T_turbine_in - T_ambient) * 0.5
    q_fuel = 1000.0 * (T_turbine_in - T_preheat) / T_turbine_in
    w_net = q_fuel * eta_turbine
    T_exhaust = T_turbine_in * (1 - eta_turbine) * (1 - 0.3 * f_recovery)
    sfc = q_fuel / max(w_net, 1e-6) * 3600
    return {"fuel_consumption": sfc, "net_power": w_net, "T_exhaust": T_exhaust, "eta_system": eta_turbine * (1 + 0.15 * f_recovery), "q_fuel": q_fuel}

In [ ]:
def objective(x):
    T_in, f_rec = x
    return system_model(T_in, f_rec)["fuel_consumption"]

def exhaust_temp_constraint(x):
    T_in, f_rec = x
    return system_model(T_in, f_rec)["T_exhaust"] - 400.0

bounds = [(600.0, 1200.0), (0.0, 0.8)]
result = minimize(objective, x0=[800.0, 0.3], bounds=bounds, constraints={"type": "ineq", "fun": exhaust_temp_constraint}, method="SLSQP")

print(f"Optimization {'converged' if result.success else 'FAILED'}")
print(f"Optimal T_turbine_in = {result.x[0]:.1f} K ({result.x[0]-273.15:.1f} C)")
print(f"Optimal f_recovery   = {result.x[1]:.3f}")
print(f"Min fuel consumption = {result.fun:.1f} kJ/kWh")

In [ ]:
T_range = np.linspace(600, 1200, 60)
f_range = np.linspace(0.0, 0.8, 50)
T_grid, F_grid = np.meshgrid(T_range, f_range)
SFC_grid = np.zeros_like(T_grid)

for i in range(len(f_range)):
    for j in range(len(T_range)):
        SFC_grid[i, j] = objective([T_grid[i, j], F_grid[i, j]])

fig = go.Figure()
fig.add_trace(go.Contour(x=T_range - 273.15, y=f_range, z=SFC_grid, colorscale="RdYlGn_r", colorbar=dict(title="SFC [kJ/kWh]"), contours=dict(showlabels=True)))
fig.add_trace(go.Scatter(x=[result.x[0] - 273.15], y=[result.x[1]], mode="markers+text", marker=dict(size=15, color="red", symbol="star"), text=["Optimum"], textposition="top center", name="Optimal Point"))
fig.update_layout(title="Fuel Consumption Landscape", xaxis_title="Turbine Inlet Temperature [C]", yaxis_title="Heat Recovery Fraction [-]", width=800, height=600)
fig.show()

In [ ]:
rows = []
for T_in in [700, 800, 900, 1000, 1100]:
    for f_rec in [0.0, 0.2, 0.4, 0.6]:
        r = system_model(float(T_in), f_rec)
        rows.append({"T_in [K]": T_in, "T_in [C]": T_in - 273.15, "f_recovery": f_rec, "SFC [kJ/kWh]": round(r["fuel_consumption"], 1), "eta_system": round(r["eta_system"], 4), "T_exhaust [K]": round(r["T_exhaust"], 1)})
df = pd.DataFrame(rows)
df

## Results

The optimizer finds the operating point that minimizes fuel consumption while maintaining exhaust temperature above the materials limit. Higher turbine inlet temperatures improve efficiency but are bounded by material limits. Heat recovery reduces fuel consumption but too much lowers exhaust temperature.